In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/blessingtosin/cc-approvals-dataset/cc_approvals.data


In [2]:
# Loading dataset
cc_app = pd.read_csv("/kaggle/input/datasets/blessingtosin/cc-approvals-dataset/cc_approvals.data")
cc_app.head()

,b,30.83,0.0,u,g,w,v,1.25,t,t.1,1,g.1,0,+
0,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
1,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
2,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
3,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+
4,b,32.08,4.000,u,g,m,v,2.50,t,f,0,g,0,+


In [3]:
# Checking data information
cc_app.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 689 entries, 0 to 688
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   b       689 non-null    object 
 1   30.83   689 non-null    object 
 2   0.0     689 non-null    float64
 3   u       689 non-null    object 
 4   g       689 non-null    object 
 5   w       689 non-null    object 
 6   v       689 non-null    object 
 7   1.25    689 non-null    float64
 8   t       689 non-null    object 
 9   t.1     689 non-null    object 
 10  1       689 non-null    int64  
 11  g.1     689 non-null    object 
 12  0       689 non-null    int64  
 13  +       689 non-null    object 
dtypes: float64(2), int64(2), object(10)
memory usage: 75.5+ KB


In [4]:
# Checking for missing values
cc_app.isnull().sum()

b        0
30.83    0
0.0      0
u        0
g        0
w        0
v        0
1.25     0
t        0
t.1      0
1        0
g.1      0
0        0
+        0
dtype: int64

In [5]:
# Making s copy fron the original dataset
df = cc_app.copy()
df.head()

,b,30.83,0.0,u,g,w,v,1.25,t,t.1,1,g.1,0,+
0,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
1,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
2,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
3,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+
4,b,32.08,4.000,u,g,m,v,2.50,t,f,0,g,0,+


In [6]:
# Seperating the dataset into categorical columns and numetical columns
cat_cols = []
num_cols = []

for col in df.columns:
    if df[col].dtype == 'object':
        cat_cols.append(col)
    else:
        num_cols.append(col) 
print("Data appending is complete")

Data appending is complete


In [7]:
# Encoding categorical xolumns

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col]) 
print(df[col])

0      0
1      0
2      0
3      0
4      0
      ..
684    1
685    1
686    1
687    1
688    1
Name: +, Length: 689, dtype: int64


In [8]:
# scaling numetical columns
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
print(df[num_cols])

          0.0      1.25         1         0
0   -0.061435  0.243606  0.739920 -0.088074
1   -0.857438 -0.216602 -0.493976 -0.037402
2   -0.648387  0.455780  0.534270 -0.194985
3    0.172742 -0.153847 -0.493976 -0.195561
4   -0.153900  0.082234 -0.493976 -0.195561
..        ...       ...       ...       ...
684  1.069251 -0.291312 -0.493976 -0.195561
685 -0.807185 -0.067184 -0.082678 -0.119936
686  1.755703 -0.067184 -0.288327 -0.195369
687 -0.916736 -0.652904 -0.493976 -0.051605
688 -0.279532  1.812500 -0.493976 -0.195561

[689 rows x 4 columns]


In [9]:
# Splitting into features and target
X = df[num_cols + cat_cols].drop(columns=['+'])
y = df['+']

print("Data split into features and target successfully")

Data split into features and target successfully


In [10]:
X.head()

,0.0,1.25,1,0,b,30.83,u,g,w,v,t,t.1,g.1
0,-0.061435,0.243606,0.739920,-0.088074,1,327,2,1,11,4,1,1,0
1,-0.857438,-0.216602,-0.493976,-0.037402,1,89,2,1,11,4,1,0,0
2,-0.648387,0.455780,0.534270,-0.194985,2,125,2,1,13,8,1,1,0
3,0.172742,-0.153847,-0.493976,-0.195561,2,43,2,1,13,8,1,0,2
4,-0.153900,0.082234,-0.493976,-0.195561,2,167,2,1,10,8,1,0,0


In [11]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: +, dtype: int64

**DATA SPLITING (TRAIN AND TEST)**

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)

print("☑️ Done")

☑️ Done


**BUILDING PIPELINE FOR EACH MODEL**

In [13]:

logistic_pipeline =  Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
]) 

decision_tree_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier())
]) 

random_forest_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
]) 

print("☑️ Done")

☑️ Done


**HYPERPARAMETER TUNING**

In [14]:
# Hyperparameter tuning to find the best setting for each model

logistic_params = {
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.01, 0.1, 1],
    'model__solver': ['liblinear']
}

decision_tree_params = {
    'model__max_depth': [3, 5],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
} 

random_forest_params = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [5, 10]
} 
print("☑️Done")

☑️Done


**HYPERPARAMETER TUNING WITH GRIDSEARCHCV**

In [15]:
# Performing Hyperparameter tuning and finding the best parameters using GridSearchCV

logistic_grid = GridSearchCV(
    logistic_pipeline, 
    logistic_params,
    cv=5, 
    error_score='raise'
) 
best_logistic = logistic_grid.fit(X_train, y_train)
logistic = logistic_grid.best_params_ 
 

print(f"Best Logistic Parameter: {logistic}")

Best Logistic Parameter: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'liblinear'}


In [16]:
# Testing and making prediction with the model (Logistic Regression) after GridSearchCV Tunning

y_pred_log = best_logistic.predict(X_test)
print(y_pred_log)

[1 1 0 1 0 0 0 1 1 1 1 0 0 1 1 1 1 1 0 0 0 0 0 1 0 1 1 0 0 1 1 0 1 1 1 0 1
 1 0 0 0 0 1 0 1 1 0 1 0 0 1 0 1 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 0 1 0 1 0
 1 0 1 1 1 0 1 0 0 0 0 1 1 0 0 0 1 1 0 1 0 0 0 1 0 0 0 1 0 0 1 0 1 1 1 0 0
 1 1 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 1 1 0 1 1 0 0 1 1 0]


In [17]:
# Performing Hyperparameter tuning and finding the best parameters using GridSearchCV

decision_tree_grid = GridSearchCV(
    decision_tree_pipeline, 
    decision_tree_params,
    cv=5, 
    error_score='raise'
) 
best_decision_tree = decision_tree_grid.fit(X_train, y_train)
decision_tree= decision_tree_grid.best_params_ 
 

print(f"Best Random Forest Parameter: {decision_tree}")

Best Random Forest Parameter: {'model__max_depth': 5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2}


In [18]:
# Testing and making prediction with the model (decision tree) after GridSearchCV Tunning

y_pred_decision = decision_tree_grid.predict(X_test)
print(y_pred_decision)

[1 1 0 1 0 0 0 1 1 1 1 0 1 1 1 1 1 1 0 0 0 0 0 1 0 1 1 0 0 1 1 0 1 1 1 0 1
 1 0 0 0 0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 0 0 1 1 0 0 0 1 1 1 1 1 1 0 1 0 1 0
 1 0 1 1 1 0 0 1 0 0 0 1 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 1 0 0 1 0 0 1 1 1 0
 1 1 0 1 1 0 0 0 1 0 1 0 1 0 1 1 0 1 1 1 0 1 1 0 1 1 0]


In [19]:
# Performing Hyperparameter tuning and finding the best parameters using GridSearchCV

random_forest_grid = GridSearchCV(
    random_forest_pipeline, 
    random_forest_params,
    cv=5, 
    error_score='raise'
) 
best_random_forest = random_forest_grid.fit(X_train, y_train)
random_forest = random_forest_grid.best_params_ 
 

print(f"Best Random Forest Parameter: {random_forest}")

Best Random Forest Parameter: {'model__max_depth': 5, 'model__n_estimators': 50}


In [20]:
# Testing and making prediction with the model (Random Forest) after GridSearchCV Tunning

y_pred_random = random_forest_grid.predict(X_test)
print(y_pred_random)

[1 1 0 1 0 0 0 1 1 1 1 0 0 1 1 1 1 1 0 0 0 0 0 1 0 1 1 0 0 1 1 0 1 1 1 1 1
 1 0 0 0 0 1 0 1 1 0 1 0 0 1 1 1 0 1 1 1 0 1 1 0 0 0 1 1 1 1 1 1 0 1 0 1 0
 1 1 1 1 1 0 1 0 0 1 0 1 1 0 0 0 1 1 0 1 1 0 0 1 0 0 0 1 0 0 1 0 0 1 1 0 0
 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 0 1 1 1 1 1 1 0 1 1 0]


**MODEL EVALUATION**

In [21]:
# Logistic Regression Model Evaluation
log_acc = accuracy_score(y_test, y_pred_log) 
log_precision = precision_score(y_test, y_pred_log) 
log_recall = recall_score(y_test, y_pred_log) 
log_f1 = f1_score(y_test, y_pred_log) 
log_cm = confusion_matrix(y_test, y_pred_log) 

print("☑️ Done")

☑️ Done


In [22]:
# Decision Tree Model Evaluation

decision_acc = accuracy_score(y_test, y_pred_decision) 
decision_precision = precision_score(y_test, y_pred_decision) 
decision_recall = recall_score(y_test, y_pred_decision) 
decision_f1 = f1_score(y_test, y_pred_decision) 
decision_cm = confusion_matrix(y_test, y_pred_decision)

print("☑️ Done")

☑️ Done


In [23]:
# Random Forest Model Evaluation

random_acc = accuracy_score(y_test, y_pred_random) 
random_precision = precision_score(y_test, y_pred_random) 
random_recall = recall_score(y_test, y_pred_random) 
random_f1 = f1_score(y_test, y_pred_random) 
random_cm = confusion_matrix(y_test, y_pred_random)

print("☑️ Done")

☑️ Done


In [24]:
# Printing the result for the model evaluation
print("="*20)
print("MODEL EVALUATION")
print("="*20)
print("LOGISTIC REGRESSION")
print(f"Accuracy: {log_acc:.4f}")
print(f"Precision: {log_precision:.4f}") 
print(f"Recall: {log_recall:.4f}") 
print(f"F1-Score: {log_f1:.4f}")
print(f"Confusion Matrix:") 
print(log_cm)
print("-"*20)

print("DECISION TREE CLASSIFIER")
print(f"Accuracy: {decision_acc:.4f}")
print(f"Precision: {decision_precision:.4f}") 
print(f"Recall: {decision_recall:.4f}") 
print(f"F1-Score: {decision_f1:.4f}")
print(f"Confusion Matrix:") 
print(decision_cm)
print("-"*20)

print("RANDOM FOREST CLASSIFIER")
print(f"Accuracy: {random_acc:.4f}")
print(f"Precision: {random_precision:.4f}") 
print(f"Recall: {random_recall:.4f}") 
print(f"F1-Score: {random_f1:.4f}")
print(f"Confusion Matrix:") 
print(random_cm)

MODEL EVALUATION
LOGISTIC REGRESSION
Accuracy: 0.7971
Precision: 0.8592
Recall: 0.7722
F1-Score: 0.8133
Confusion Matrix:
[[49 10]
 [18 61]]
--------------------
DECISION TREE CLASSIFIER
Accuracy: 0.8188
Precision: 0.8649
Recall: 0.8101
F1-Score: 0.8366
Confusion Matrix:
[[49 10]
 [15 64]]
--------------------
RANDOM FOREST CLASSIFIER
Accuracy: 0.7971
Precision: 0.8228
Recall: 0.8228
F1-Score: 0.8228
Confusion Matrix:
[[45 14]
 [14 65]]


**BEST MODEL WITH ACCURACY**

In [25]:
# Creating dictionary of accuracies
accuracies = {
    'Logistic Regression': log_acc,
    'Decision Tree': decision_acc,
    'Random Forest': random_acc
}
print("☑️ Done")

☑️ Done


In [26]:
# Checking for the best model and accuraxy

best_model_name = max(accuracies, key=accuracies.get) 
best_accuracy = accuracies[best_model_name] 

if best_model_name == 'Logistic Regression':
    best_model = logistic_grid.best_estimator_
elif best_model_name == 'Decision Tree':
    best_model = decision_tree_grid.best_estimator_ 
else:
    best_model = random_forest_grid.best_estimator_ 

print(f"BEST MODEL: {best_model_name}")
print(f"BEST ACCURACY: {best_accuracy:.4f}") 
print(f"BEST MODRL = {best_model}")
print(f"BEST SCORE = {best_accuracy:.4f}") 


if best_accuracy >= 0.75:
    print(f"Requirement met (Accuracy >= 0.75).") 
else:
    print(f"Does not met the required value.")

BEST MODEL: Decision Tree
BEST ACCURACY: 0.8188
BEST MODRL = Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 DecisionTreeClassifier(max_depth=5, min_samples_leaf=2))])
BEST SCORE = 0.8188
Requirement met (Accuracy >= 0.75).
